# Loading and Running a Pre-trained LLM

In [ ]:
import jax
import flax.nnx as nnx
from pathlib import Path

from helper import MiniGPT, generate_story

In [ ]:
model = MiniGPT()

In [ ]:
model

## Load the saved checkpoint

In [ ]:
import orbax
from orbax import checkpoint

from jax.sharding import SingleDeviceSharding 

In [ ]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

In [ ]:
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

In [ ]:
nnx.state(model)

In [ ]:
checkpoint_path = Path.cwd() / "model_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [ ]:
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model,restored_state)

## Run inference

In [ ]:
def create_story(story_prompt, temperature, max_new_tokens):
    return generate_story(model, story_prompt, temperature, max_new_tokens)

In [ ]:
create_story("Once upon a time a big bear ", 0.2, 30)

## Interactive chat demo

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=create_story,
    inputs=[
        gr.Textbox(label="Story Prompt"),         
        gr.Slider(
            minimum=0, maximum=1.0, value=0.8, step=0.01, label="Temperature"
        ),
        gr.Slider(minimum=0, maximum=200, value=10, step=1, label="Max Tokens"
        )
    ],
    outputs=["text"]
)

demo.launch(share=True)